# Phase 8: Evaluation & Metrics

**Pipeline position:**
```
Phase 1–7: Full Pipeline  ✓
Phase 8: Evaluation       ← YOU ARE HERE
```

---

## Why Quantitative Evaluation?

> "It looks good" is not a metric.

Without numbers, you cannot:
- Know if a parameter change actually improved anything
- Compare this pipeline to a different one
- Report results in a paper or portfolio
- Debug which phase is causing the most error

Because we used **synthetic data with ground truth**, every output of the pipeline can be compared to a known-correct reference.  This is the principal advantage of the synthetic setup: it turns qualitative inspection into quantitative science.

---

## The Three Metrics

### 1. Reprojection Error (Phase 3 — SfM quality)

After bundle adjustment, project every reconstructed 3-D point back through its associated cameras and measure the pixel distance to the original 2-D observation:

$$e_{\text{reproj}} = \frac{1}{N} \sum_{c,j} \left\| \pi\bigl(K[R_c|\mathbf{t}_c]\mathbf{X}_j\bigr) - \mathbf{x}_{cj} \right\|_2 \quad [\text{pixels}]$$

**Interpretation:**
- $< 1$ px — excellent
- $1$–$2$ px — good (acceptable for most photogrammetry)
- $> 2$ px — SfM or BA converged poorly; check matches and geometry

This measures **internal consistency** of the SfM solution, not accuracy vs. the real world.  A low reprojection error does not guarantee metric accuracy — it only means the poses and points are mutually consistent.

### 2. Chamfer Distance (Phases 4–5 — Dense reconstruction quality)

Compares the dense predicted point cloud to a ground-truth cloud (built by back-projecting GT depth maps):

$$d_{\text{Chamfer}}(S_1, S_2) = \frac{1}{|S_1|}\sum_{\mathbf{x} \in S_1} \min_{\mathbf{y} \in S_2} \|\mathbf{x} - \mathbf{y}\|_2 \;+\; \frac{1}{|S_2|}\sum_{\mathbf{y} \in S_2} \min_{\mathbf{x} \in S_1} \|\mathbf{x} - \mathbf{y}\|_2 \quad [\text{metres}]$$

Both terms matter:
- First term (S₁→S₂): measures **completeness** — are all GT points covered by predictions?
- Second term (S₂→S₁): measures **accuracy** — are all predicted points close to reality?

**Interpretation:** $< 0.5$ m is excellent for a 50 m altitude, 100 m scene.  $> 2$ m suggests the depth estimation is significantly wrong.

### 3. Pose Error (Phase 3 — Direct GT comparison)

Since we have exact GT poses from Phase 1, we can measure the angular error of each estimated camera rotation:

$$\theta_{err} = \arccos\!\left(\frac{\text{tr}(R_{est}^\top R_{GT}) - 1}{2}\right)$$

This is the **geodesic distance** on SO(3) — the minimum rotation angle needed to go from $R_{est}$ to $R_{GT}$.

---

## How to Use Metrics to Debug

```
High reprojection error   →  Bad matches or BA didn't converge
                             Fix: more features, tighter ratio test, more BA iterations

Low reproj, high Chamfer  →  Good poses but bad depth estimation (MVS)
                             Fix: more stereo neighbors, larger num_disparities

High pose error           →  Scale drift or degenerate initial pair
                             Fix: better initial pair selection, GPS anchoring

Everything bad            →  Low feature coverage, check match graph connectivity
```

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
import numpy as np
import cv2
import open3d as o3d
import matplotlib.pyplot as plt
import json
%matplotlib inline

In [ ]:
outputs_dir = os.path.join('..', 'outputs')
poses_path  = os.path.join('..', 'data', 'synthetic', 'GT_poses.json')
img_dir     = os.path.join('..', 'data', 'synthetic', 'images')
depth_dir   = os.path.join('..', 'data', 'synthetic', 'GT_depth')
assert os.path.exists(poses_path), "Run phase1_data_generation.ipynb first."

with open(poses_path) as f:
    gt_poses = json.load(f)

K = np.array(gt_poses[0]['K'])
img_files = sorted(os.listdir(img_dir))
images    = [cv2.imread(os.path.join(img_dir, f)) for f in img_files]

# Load any pre-computed metrics
metrics_path = os.path.join(outputs_dir, 'metrics.json')
metrics = {}
if os.path.exists(metrics_path):
    with open(metrics_path) as f:
        metrics = json.load(f)
    print("Loaded pre-computed metrics:")
    print(json.dumps(metrics, indent=2))
else:
    print("No pre-computed metrics found — will compute from scratch below.")

---

## Metric 1 — SfM Reprojection Error

We run SfM on **8 images** (subset for speed) and compute reprojection errors before and after Bundle Adjustment.  The improvement from BA demonstrates how much incremental drift BA corrects.

> **Connection to Phase 3:**  
> This mini SfM run is a self-contained demo — it does not load Phase 3's output.  
> Phase 3 saved its full 16-camera estimated poses to `outputs/estimated_poses.json`.  
> **Metric 2 (Pose Error)** below will load those poses if the file exists, giving a true  
> assessment of the complete Phase 3 pipeline rather than this 8-image subset.

In [ ]:
from src.features.detector import detect_features
from src.features.matcher import match_features
from src.sfm.incremental import run_incremental_sfm
from src.sfm.bundle_adjustment import run_bundle_adjustment
from src.evaluation.metrics import reprojection_error

n_eval = 8   # use 8 cameras for reasonable speed
print(f"Running SfM on {n_eval} cameras...")
feats = detect_features(images[:n_eval], nfeatures=1024)
mgraph = match_features(feats, ratio=0.75, min_inliers=20)
recon = run_incremental_sfm(mgraph, K)

# Pre-BA reprojection error
def compute_reproj_errors(recon, K):
    errs = []
    for (cam_id, pt_id), pt2d in recon.observations.items():
        R, t = recon.cameras[cam_id]
        X = recon.points3d[pt_id].reshape(1, 3)
        rvec, _ = cv2.Rodrigues(R)
        proj, _ = cv2.projectPoints(X, rvec, t, K, None)
        errs.append(np.linalg.norm(proj.ravel() - pt2d))
    return np.array(errs)

errors_pre_ba = compute_reproj_errors(recon, K)

print(f"\nBefore BA: mean={errors_pre_ba.mean():.3f} px, median={np.median(errors_pre_ba):.3f} px")

ba_error = run_bundle_adjustment(recon, max_iter=20)
errors_post_ba = compute_reproj_errors(recon, K)

print(f"After  BA: mean={errors_post_ba.mean():.3f} px, median={np.median(errors_post_ba):.3f} px")
print(f"Improvement: {errors_pre_ba.mean() - errors_post_ba.mean():.3f} px ({100*(1-errors_post_ba.mean()/errors_pre_ba.mean()):.1f}% reduction)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, errors, label, color in [
    (axes[0], errors_pre_ba,  'Before BA', 'orange'),
    (axes[1], errors_post_ba, 'After BA',  'steelblue')
]:
    ax.hist(errors, bins=60, color=color, edgecolor='none', alpha=0.8)
    ax.axvline(errors.mean(), color='red', linestyle='--',
               label=f'mean = {errors.mean():.2f} px')
    ax.axvline(1.0, color='green', linestyle=':', label='1 px target')
    ax.axvline(2.0, color='orange', linestyle=':', label='2 px threshold')
    ax.set_xlabel('Reprojection error (px)')
    ax.set_ylabel('Count')
    ax.set_title(f'Reprojection Error — {label}')
    ax.legend(fontsize=8)

plt.suptitle('Effect of Bundle Adjustment on Reprojection Error', fontsize=12)
plt.tight_layout()
plt.show()

repr_metrics = reprojection_error(recon, K)
metrics['reprojection'] = repr_metrics
print("Reprojection metrics:", repr_metrics)

---

## Metric 2 — Pose Error vs. Ground Truth

Unlike reprojection error (which is internal consistency), pose error measures how close our estimated camera orientations are to the **true** physical poses.  This requires ground truth — impossible in the real world without GPS or surveyed control points, but available here because we generated the data synthetically.

Note that pose errors typically **increase** for later-registered cameras due to error accumulation, and BA only partially corrects this.

In [ ]:
# Load Phase 3 full estimated poses if available; fall back to the 8-camera demo above.
est_poses_path = os.path.join(outputs_dir, 'estimated_poses.json')

if os.path.exists(est_poses_path):
    with open(est_poses_path) as f:
        est_poses_list = json.load(f)
    eval_cameras = {p['cam_id']: (np.array(p['R']), np.array(p['t']))
                    for p in est_poses_list}
    source_label = f"Phase 3 full pipeline ({len(eval_cameras)} cameras)"
    print(f"Loaded Phase 3 estimated poses: {len(eval_cameras)} cameras")
else:
    eval_cameras = dict(recon.cameras)
    source_label = f"8-camera SfM demo ({len(eval_cameras)} cameras)"
    print("outputs/estimated_poses.json not found.")
    print("Using 8-camera SfM demo poses. Run phase3_sfm.ipynb for full evaluation.")

# Rotation error: geodesic distance on SO(3) between estimated and GT rotation.
angle_errors = {}
for cam_id, (R_est, t_est) in eval_cameras.items():
    if cam_id >= len(gt_poses):
        continue
    R_gt  = np.array(gt_poses[cam_id]['R'])
    R_err = R_est @ R_gt.T
    cos_a = np.clip((np.trace(R_err) - 1) / 2, -1, 1)
    angle_errors[cam_id] = np.degrees(np.arccos(cos_a))

cam_ids = sorted(angle_errors.keys())
errs    = [angle_errors[c] for c in cam_ids]

fig, ax = plt.subplots(figsize=(10, 4))
colors = ['green' if e < 2 else 'orange' if e < 5 else 'red' for e in errs]
ax.bar(range(len(cam_ids)), errs, color=colors, alpha=0.85)
ax.axhline(2.0, color='orange', linestyle='--', label='2° threshold (good)')
ax.axhline(5.0, color='red',    linestyle='--', label='5° threshold (marginal)')
ax.set_xticks(range(len(cam_ids)))
ax.set_xticklabels([f'Cam {c}' for c in cam_ids], rotation=45, ha='right')
ax.set_ylabel('Rotation error (°)')
ax.set_title(f'Per-Camera Rotation Error vs. GT — {source_label}\n'
             'Green < 2°, Orange 2–5°, Red > 5°')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Source: {source_label}")
print(f"Rotation error summary:")
print(f"  Mean   : {np.mean(errs):.3f}°")
print(f"  Max    : {np.max(errs):.3f}°  (camera {cam_ids[np.argmax(errs)]})")
print(f"  Good (< 2°): {sum(e < 2 for e in errs)} / {len(errs)} cameras")

---

## Metric 3 — Chamfer Distance (Dense Reconstruction)

We build a GT point cloud by back-projecting the ground-truth depth maps (Phase 1).  Then we compare it to the dense predicted cloud (Phase 4).

**Important:** Chamfer distance is computed after **voxel downsampling** of both clouds to the same resolution.  Without downsampling, a denser predicted cloud would artificially have lower error in the first term (every GT point has a nearby predicted point even if predictions are noisy).

In [ ]:
from src.evaluation.metrics import chamfer_distance
from src.mvs.depth_fusion import depth_to_pointcloud

dense_path = os.path.join(outputs_dir, 'dense_cloud.ply')
if not os.path.exists(dense_path):
    dense_path = os.path.join(outputs_dir, 'dense_cloud_subset.ply')

if os.path.exists(dense_path):
    pcd_pred = o3d.io.read_point_cloud(dense_path)

    # Build GT cloud from Phase 1 GT depth maps (sample 4 views for speed)
    depth_files = sorted(os.listdir(depth_dir))[:4]
    pcd_gt = o3d.geometry.PointCloud()
    for i, df in enumerate(depth_files):
        d = np.load(os.path.join(depth_dir, df))
        R = np.array(gt_poses[i]['R'])
        t = np.array(gt_poses[i]['t'])
        pcd_i = depth_to_pointcloud(d, images[i], K, R, t, max_depth=80.0)
        pcd_gt += pcd_i
    pcd_gt = pcd_gt.voxel_down_sample(0.5)

    cd_metrics = chamfer_distance(pcd_pred, pcd_gt)
    metrics['chamfer'] = cd_metrics

    print(f"Predicted cloud: {len(pcd_pred.points):,} points")
    print(f"GT cloud (4 views, 0.5m voxel): {len(pcd_gt.points):,} points")
    print(f"\nChamfer metrics: {cd_metrics}")
else:
    print("Dense cloud not found. Run phase4_mvs.ipynb first.")
    cd_metrics = {}
    pcd_pred = pcd_gt = None

In [ ]:
if pcd_pred is not None and pcd_gt is not None:
    pts_pred = np.asarray(pcd_pred.points)
    pts_gt   = np.asarray(pcd_gt.points)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Top view overlay
    axes[0].scatter(pts_gt[:,0], pts_gt[:,1], s=0.4, c='limegreen', alpha=0.4, label='GT')
    axes[0].scatter(pts_pred[:,0], pts_pred[:,1], s=0.4, c='steelblue', alpha=0.4, label='Predicted')
    axes[0].set_title('Top View: GT vs. Predicted\nGreen = GT, Blue = Predicted')
    axes[0].set_xlabel('X (m)'); axes[0].set_ylabel('Y (m)')
    axes[0].legend(markerscale=5, fontsize=8)
    axes[0].set_aspect('equal')

    # Side view
    axes[1].scatter(pts_pred[:,0], pts_pred[:,2], s=0.4, c='steelblue', alpha=0.4)
    axes[1].set_title('Predicted Cloud — Side View (X-Z)\nShould show ground + building heights')
    axes[1].set_xlabel('X (m)'); axes[1].set_ylabel('Z (m)')

    chamfer_val = cd_metrics.get('chamfer_mean', 'N/A')
    plt.suptitle(f'Dense Reconstruction Accuracy  |  Chamfer distance = {chamfer_val}',
                 fontsize=12)
    plt.tight_layout()
    plt.show()

---

## Final Summary: Pipeline Scorecard

In [ ]:
# Save all metrics
metrics_out = os.path.join(outputs_dir, 'metrics.json')
os.makedirs(outputs_dir, exist_ok=True)
with open(metrics_out, 'w') as f:
    json.dump(metrics, f, indent=2)

# Print scorecard
print("=" * 55)
print("  AERIAL PHOTOGRAMMETRY PIPELINE — SCORECARD")
print("=" * 55)

# Reprojection
r = metrics.get('reprojection', {})
mean_px = r.get('mean_px', errors_post_ba.mean() if 'errors_post_ba' in locals() else None)
if mean_px is not None:
    status = '✓ Excellent' if mean_px < 1 else '✓ Good' if mean_px < 2 else '✗ Poor'
    print(f"  Reprojection error   : {mean_px:.3f} px    {status}")
    print(f"    Target: < 2 px")

# Chamfer
c = metrics.get('chamfer', {})
cval = c.get('chamfer_mean', None)
if cval is not None:
    status = '✓ Excellent' if cval < 0.5 else '✓ Good' if cval < 1.0 else '✗ Poor'
    print(f"  Chamfer distance     : {cval:.4f} m    {status}")
    print(f"    Target: < 1 m (50 m altitude, 100 m scene)")

# Pose
if 'errs' in locals() and len(errs):
    mean_rot = np.mean(errs)
    status = '✓ Excellent' if mean_rot < 2 else '✓ Good' if mean_rot < 5 else '✗ Poor'
    print(f"  Mean rotation error  : {mean_rot:.3f}°    {status}")
    print(f"    Target: < 5°")

print("=" * 55)
print(f"  Metrics saved: {metrics_out}")

In [ ]:
# Visual comparison: one rendered image vs. one GT image
# Shows whether the texture atlas looks like the source photography
atlas_orig    = os.path.join(outputs_dir, 'texture_atlas.png')
atlas_painted = os.path.join(outputs_dir, 'texture_atlas_inpainted.png')

if os.path.exists(atlas_orig) and os.path.exists(atlas_painted):
    a_orig    = cv2.cvtColor(cv2.imread(atlas_orig),    cv2.COLOR_BGR2RGB)
    a_painted = cv2.cvtColor(cv2.imread(atlas_painted), cv2.COLOR_BGR2RGB)

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    axes[0].imshow(cv2.cvtColor(images[0], cv2.COLOR_BGR2RGB))
    axes[0].set_title('Source Photography (Image 0)')
    axes[0].axis('off')

    axes[1].imshow(a_orig)
    axes[1].set_title('Texture Atlas (before inpainting)')
    axes[1].axis('off')

    axes[2].imshow(a_painted)
    axes[2].set_title('Texture Atlas (after inpainting)')
    axes[2].axis('off')

    plt.suptitle('Pipeline Output: Photography → Texture Atlas', fontsize=12)
    plt.tight_layout()
    plt.show()

---

## Pipeline Complete

```
Phase 1 ─ Synthetic Scene + Camera Rig
           └─ 16 aerial images, GT poses, GT depth maps
Phase 2 ─ SIFT + FLANN + RANSAC
           └─ Match graph: verified 2D correspondences
Phase 3 ─ Essential Matrix + PnP + Bundle Adjustment
           └─ Sparse 3D cloud + calibrated camera poses
Phase 4 ─ Stereo SGBM + Multi-View Fusion
           └─ Dense depth maps + dense point cloud
Phase 5 ─ Poisson Surface Reconstruction
           └─ Watertight mesh (80K faces)
Phase 6 ─ View Selection + UV Unwrapping + Projection
           └─ 2048×2048 texture atlas
Phase 7 ─ Occlusion Detection + Telea Inpainting
           └─ Complete, hole-free texture atlas
Phase 8 ─ Reprojection Error + Chamfer Distance + Pose Error
           └─ Quantitative assessment of the full pipeline
```

---

## Where to Go From Here

### On Real Data
1. Capture imagery with a DJI Mavic or similar — at least 80% overlap between frames
2. Use GPS-tagged images for metric scale (remove the scale ambiguity in Phase 3)
3. Add ground control points (GCPs) for absolute accuracy < 10 cm

### Algorithm Improvements
- Replace SIFT with SuperPoint + SuperGlue (neural feature matching — more robust on textureless facades)
- Replace SGBM with MVSNet or PatchMatchNet (learned MVS — much higher coverage)
- Replace simple UV packing with angle-based flattening (xatlas) — reduces texture distortion
- Replace Telea with LaMa for large occlusions

### Things to Experiment With
- How does camera overlap (change `grid_size` in Phase 1) affect Chamfer distance?
- How does altitude affect reprojection error and texture resolution simultaneously?
- Run the full pipeline twice with different `seed` values for the synthetic scene — how stable are the metrics?

---

**You have built a complete aerial photogrammetry pipeline from scratch.**